# Task 2: Deep Learning - Image Classification with Transfer Learning (PyTorch)
Pretrained **ResNet-18** (ImageNet) fine-tuned on a **CIFAR-10 subset** (5,000 train / 1,000 val / 1,000 test images).
**Run:** Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all (~5-8 min; CIFAR-10 downloads automatically). It also runs on CPU, just much slower.

Covers: transfer learning (freeze backbone -> train new head -> fine-tune layer4), data augmentation, training curves, evaluation metrics, saved model, inference script and a live inference demo.
Outputs: `figures/task2_curves.png`, `figures/task2_cm.png`, `figures/task2_samples.png`, `cifar10_resnet18.pth`, `predict.py`, `README_inference.md`, `Task2_Report.md`.

In [ ]:
import os, random, numpy as np, torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt, seaborn as sns
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
os.makedirs('figures', exist_ok=True)
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('Device:', dev)
if dev.type=='cpu': print('WARNING: no GPU found - training will be slow. Use Runtime > Change runtime type > T4 GPU.')

## 1. Data: CIFAR-10 subset + augmentation
Augmentation (training only): resize to 128, random crop with padding, horizontal flip, rotation +/-10 degrees. ImageNet mean/std normalisation because the backbone was pretrained on ImageNet.

In [ ]:
PER_CLASS_TRAIN, PER_CLASS_TEST = 500, 100          # subset size -> 5000 train, 1000 test images
mean, std = (0.485,0.456,0.406), (0.229,0.224,0.225)  # ImageNet statistics
train_tf = transforms.Compose([transforms.Resize(128), transforms.RandomCrop(128, padding=8),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10), transforms.ToTensor(), transforms.Normalize(mean,std)])
eval_tf  = transforms.Compose([transforms.Resize(128), transforms.ToTensor(), transforms.Normalize(mean,std)])
full_tr = datasets.CIFAR10('data', train=True,  download=True, transform=train_tf)
full_ev = datasets.CIFAR10('data', train=True,  download=True, transform=eval_tf)
full_te = datasets.CIFAR10('data', train=False, download=True, transform=eval_tf)
classes = full_tr.classes
def pick(ds, per_class, offset=0):
    idx=[]; t=np.array(ds.targets)
    for c in range(10): idx += np.where(t==c)[0][offset:offset+per_class].tolist()
    return idx
tr_idx  = pick(full_tr, PER_CLASS_TRAIN)
val_idx = pick(full_tr, 100, offset=PER_CLASS_TRAIN)     # disjoint from the training images
te_idx  = pick(full_te, PER_CLASS_TEST)
train_dl = DataLoader(Subset(full_tr, tr_idx), batch_size=64, shuffle=True, num_workers=2)
val_dl   = DataLoader(Subset(full_ev, val_idx), batch_size=64, num_workers=2)
test_dl  = DataLoader(Subset(full_te, te_idx), batch_size=64, num_workers=2)
print(len(tr_idx), len(val_idx), len(te_idx))

## 2. Pretrained ResNet-18: freeze backbone, new head, then fine-tune
Stage 1 (5 epochs): only the new 10-class head is trained. Stage 2 (5 epochs): `layer4` is unfrozen and fine-tuned with a small learning rate.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in model.parameters(): p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(dev); crit = nn.CrossEntropyLoss()
hist = {'tr_loss':[],'va_loss':[],'tr_acc':[],'va_acc':[]}

def run_epoch(dl, train, opt=None):
    model.train(train); tot=loss_sum=correct=0
    with torch.set_grad_enabled(train):
        for x,y in dl:
            x,y = x.to(dev), y.to(dev); out = model(x); loss = crit(out,y)
            if train: opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item()*len(y); correct += (out.argmax(1)==y).sum().item(); tot += len(y)
    return loss_sum/tot, correct/tot

def fit(epochs, opt):
    for e in range(epochs):
        a = run_epoch(train_dl, True, opt); b = run_epoch(val_dl, False)
        for k,v in zip(hist, (a[0],b[0],a[1],b[1])): hist[k].append(v)
        print(f"epoch {len(hist['tr_loss']):2d} | train loss {a[0]:.3f} acc {a[1]:.3f} | val loss {b[0]:.3f} acc {b[1]:.3f}")

print('Stage 1: train head only'); fit(5, optim.Adam(model.fc.parameters(), lr=1e-3))
for p in model.layer4.parameters(): p.requires_grad = True
print('Stage 2: fine-tune layer4 + head')
fit(5, optim.Adam([{'params':model.layer4.parameters(),'lr':1e-4},{'params':model.fc.parameters(),'lr':1e-4}]))

## 3. Training curves (dashed line = start of fine-tuning)

In [ ]:
ep = range(1, len(hist['tr_loss'])+1)
fig, ax = plt.subplots(1,2, figsize=(11,4))
ax[0].plot(ep,hist['tr_loss'],label='train'); ax[0].plot(ep,hist['va_loss'],label='validation'); ax[0].set_title('Loss'); ax[0].set_xlabel('Epoch'); ax[0].legend()
ax[1].plot(ep,hist['tr_acc'],label='train'); ax[1].plot(ep,hist['va_acc'],label='validation'); ax[1].set_title('Accuracy'); ax[1].set_xlabel('Epoch'); ax[1].legend()
for a in ax: a.axvline(5.5, color='grey', ls='--', lw=0.8)
plt.tight_layout(); plt.savefig('figures/task2_curves.png', dpi=200); plt.show()

## 4. Evaluation metrics, confusion matrix, sample predictions

In [ ]:
def predict_all(dl):
    model.eval(); P=[];T=[];I=[]
    with torch.no_grad():
        for x,y in dl:
            P += model(x.to(dev)).argmax(1).cpu().tolist(); T += y.tolist(); I.append(x)
    return np.array(P), np.array(T), torch.cat(I)
R = {}
for nm,dl in [('Validation',val_dl),('Test',test_dl)]:
    p,t,_ = predict_all(dl); pr,rc,f1,_ = precision_recall_fscore_support(t,p,average='macro')
    l,_ = run_epoch(dl, False); R[nm] = dict(acc=accuracy_score(t,p)*100, P=pr, R=rc, F1=f1, loss=l)
    print(f'{nm}: acc {R[nm]["acc"]:.2f}% | macro P {pr:.4f} R {rc:.4f} F1 {f1:.4f} | loss {l:.4f}')
p,t,imgs = predict_all(test_dl)
print(classification_report(t,p,target_names=classes))
plt.figure(figsize=(7,6)); sns.heatmap(confusion_matrix(t,p), annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.savefig('figures/task2_cm.png', dpi=200); plt.show()
inv = lambda x: (x.permute(1,2,0).numpy()*np.array(std)+np.array(mean)).clip(0,1)
sel = np.random.RandomState(0).choice(len(t), 12, replace=False)
fig, ax = plt.subplots(2,6, figsize=(14,5.5))
for a,i in zip(ax.ravel(), sel):
    a.imshow(inv(imgs[i])); a.axis('off'); a.set_title(f'T:{classes[t[i]]}\nP:{classes[p[i]]}', color='green' if t[i]==p[i] else 'red', fontsize=9)
plt.tight_layout(); plt.savefig('figures/task2_samples.png', dpi=200); plt.show()

## 5. Save model, inference script and run inference

In [ ]:
torch.save({'state_dict': model.state_dict(), 'classes': classes}, 'cifar10_resnet18.pth')
PREDICT = """import argparse, torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
ap = argparse.ArgumentParser(); ap.add_argument('--image', required=True); ap.add_argument('--model', default='cifar10_resnet18.pth')
a = ap.parse_args()
ck = torch.load(a.model, map_location='cpu'); m = models.resnet18(); m.fc = nn.Linear(m.fc.in_features, len(ck['classes']))
m.load_state_dict(ck['state_dict']); m.eval()
tf = transforms.Compose([transforms.Resize(128), transforms.ToTensor(), transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
x = tf(Image.open(a.image).convert('RGB')).unsqueeze(0)
with torch.no_grad(): pr = torch.softmax(m(x),1)[0]
top = pr.topk(3)
print(f"Predicted: {ck['classes'][int(top.indices[0])]} (confidence {top.values[0]*100:.1f}%)")
print('Top-3:', ', '.join(f"{ck['classes'][int(i)]} {v*100:.1f}%" for v,i in zip(top.values, top.indices)))
"""
open('predict.py','w').write(PREDICT)
README = """# Running inference
1. Install: `pip install torch torchvision pillow`
2. Put `cifar10_resnet18.pth` and `predict.py` in one folder.
3. Run: `python predict.py --image path/to/photo.jpg`   (optional: `--model other_checkpoint.pth`). Works on CPU; no GPU needed.
Output example: `Predicted: truck (confidence 97.3%)` plus the top-3 classes.
Classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.
Images are resized to 128x128, so photos of a single, centred object work best.
"""
open('README_inference.md','w').write(README)
print('saved cifar10_resnet18.pth, predict.py, README_inference.md')

### Inference demo: run `predict.py` on a real test image, exactly as an end user would

In [ ]:
import subprocess
raw = datasets.CIFAR10('data', train=False, download=False)      # un-transformed PIL images
img, lab = raw[te_idx[7]]; img.resize((256,256)).save('sample.png'); print('True label:', classes[lab])
r = subprocess.run(['python','predict.py','--image','sample.png'], capture_output=True, text=True); print(r.stdout, r.stderr[-500:])

## 6. Short report (auto-generated)

In [ ]:
rows = chr(10).join(f"| {k} | {v['acc']:.2f}% | {v['P']:.4f} | {v['R']:.4f} | {v['F1']:.4f} | {v['loss']:.4f} |" for k,v in R.items())
report = f"""# Task 2 Report: Transfer-Learning Image Classifier (CIFAR-10 subset)

## Setup
- Data: CIFAR-10 subset, {len(tr_idx)} train / {len(val_idx)} validation / {len(te_idx)} test images (balanced, 10 classes); validation images are disjoint from training images.
- Augmentation (train only): resize 128, random crop (padding 8), horizontal flip, rotation +/-10 degrees; ImageNet normalisation.
- Model: ResNet-18 pretrained on ImageNet. Stage 1 trains only the new 10-way head (5 epochs, Adam lr 1e-3); Stage 2 also fine-tunes layer4 (5 epochs, Adam lr 1e-4).

## Results
| Split | Accuracy | Macro precision | Macro recall | Macro F1 | Loss |
|---|---|---|---|---|---|
{rows}

Training curves (figures/task2_curves.png): validation accuracy rises from {hist['va_acc'][0]*100:.0f}% (epoch 1) to {hist['va_acc'][4]*100:.0f}% after head-only training, then to {hist['va_acc'][-1]*100:.0f}% after fine-tuning layer4.
Final train accuracy is {hist['tr_acc'][-1]*100:.0f}% vs {hist['va_acc'][-1]*100:.0f}% validation, so some overfitting appears during fine-tuning (train loss keeps falling while validation loss flattens); more data, stronger augmentation or early stopping would help.
Errors concentrate on visually similar classes (e.g. cat/dog, deer/horse, airplane/ship); see figures/task2_cm.png.

## Saved model and inference
cifar10_resnet18.pth (weights + class names) and predict.py. Run: python predict.py --image photo.jpg (see README_inference.md).
"""
open('Task2_Report.md','w').write(report); print(report)

In [ ]:
try:
    from google.colab import files
    for f in ['cifar10_resnet18.pth','predict.py','README_inference.md','Task2_Report.md','figures/task2_curves.png','figures/task2_cm.png','figures/task2_samples.png']: files.download(f)
except ImportError: pass